# Topic 05. CountVectorizer와 코사인 유사도 기반 텍스트 분류

4월 과제의 핵심인 단어 빈도 벡터화와 코사인 유사도 분류를 로컬 샘플 문장으로 재구성합니다.
`fetch_20newsgroups`처럼 네트워크나 캐시 상태에 의존할 수 있는 부분은 사용하지 않습니다.


## 제출자 정보

- 이름: 이성민
- 학과: 소프트웨어융합과
- 학년: 2학년
- 학번: 2151050


## 1. 작은 뉴스 데이터셋 만들기

텍스트 분류는 문서와 라벨 쌍이 있어야 합니다.
여기서는 컴퓨터 그래픽, 우주 과학, 종교 토론을 흉내 낸 짧은 문장을 사용합니다.


In [1]:
import pandas as pd

documents = pd.DataFrame(
    [
        ("comp.graphics", "rendering pixels shader image graphics"),
        ("comp.graphics", "3d model texture screen animation"),
        ("comp.graphics", "vector raster display color image"),
        ("sci.space", "orbit rocket nasa planet mission"),
        ("sci.space", "space telescope galaxy launch satellite"),
        ("sci.space", "astronaut moon orbit science"),
        ("talk.religion.misc", "faith belief scripture community"),
        ("talk.religion.misc", "religion debate ethics belief"),
        ("talk.religion.misc", "church tradition philosophy faith"),
    ],
    columns=["category", "text"],
)

documents


,category,text
0,comp.graphics,rendering pixels shader image graphics
1,comp.graphics,3d model texture screen animation
2,comp.graphics,vector raster display color image
3,sci.space,orbit rocket nasa planet mission
4,sci.space,space telescope galaxy launch satellite
5,sci.space,astronaut moon orbit science
6,talk.religion.misc,faith belief scripture community
7,talk.religion.misc,religion debate ethics belief
8,talk.religion.misc,church tradition philosophy faith


## 2. CountVectorizer로 단어 빈도 벡터화

컴퓨터는 텍스트 자체를 바로 계산하지 못하므로 단어 출현 횟수 벡터로 바꿉니다.
이때 학습 데이터에 없던 단어는 OOV 문제가 됩니다.


In [2]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = CountVectorizer(stop_words="english")
X = vectorizer.fit_transform(documents["text"])

vocab_preview = vectorizer.get_feature_names_out()[:10].tolist()
print("vocabulary size:", len(vectorizer.vocabulary_))
print("preview:", vocab_preview)


vocabulary size: 37
preview: ['3d', 'animation', 'astronaut', 'belief', 'church', 'color', 'community', 'debate', 'display', 'ethics']


## 3. 코사인 유사도로 새 문장 분류

코사인 유사도는 두 벡터의 방향이 얼마나 비슷한지 봅니다.
새 문장과 가장 비슷한 학습 문서의 라벨을 예측 라벨로 사용합니다.


In [3]:
def classify_by_cosine(text):
    query_vector = vectorizer.transform([text])
    similarities = cosine_similarity(query_vector, X).ravel()
    best_index = int(similarities.argmax())
    return {
        "input": text,
        "predicted_category": documents.loc[best_index, "category"],
        "similarity": round(float(similarities[best_index]), 3),
        "matched_text": documents.loc[best_index, "text"],
    }


classify_by_cosine("rocket launch telescope orbit")


{'input': 'rocket launch telescope orbit',
 'predicted_category': 'sci.space',
 'similarity': 0.447,
 'matched_text': 'orbit rocket nasa planet mission'}

## 4. TF-IDF와 bigram 비교

CountVectorizer는 빈도만 보지만, TF-IDF는 흔한 단어의 영향력을 낮춥니다.
bigram은 두 단어 묶음을 함께 보므로 짧은 구문 정보를 일부 반영할 수 있습니다.


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

test_sentences = [
    ("sci.space", "satellite orbit mission"),
    ("comp.graphics", "graphics shader texture"),
    ("talk.religion.misc", "faith ethics debate"),
]

def evaluate_vectorizer(make_vectorizer):
    local_vectorizer = make_vectorizer()
    local_X = local_vectorizer.fit_transform(documents["text"])
    correct = 0
    for expected, text in test_sentences:
        query = local_vectorizer.transform([text])
        similarities = cosine_similarity(query, local_X).ravel()
        predicted = documents.loc[int(similarities.argmax()), "category"]
        correct += int(predicted == expected)
    return correct / len(test_sentences)

comparison = pd.DataFrame(
    [
        {"method": "count_unigram", "accuracy": evaluate_vectorizer(lambda: CountVectorizer(stop_words="english"))},
        {"method": "tfidf_unigram", "accuracy": evaluate_vectorizer(lambda: TfidfVectorizer(stop_words="english"))},
        {"method": "count_bigram", "accuracy": evaluate_vectorizer(lambda: CountVectorizer(stop_words="english", ngram_range=(1, 2)))},
    ]
)
comparison


,method,accuracy
0,count_unigram,1.0
1,tfidf_unigram,1.0
2,count_bigram,1.0


## 정리

- CountVectorizer는 텍스트를 단어 빈도 벡터로 바꿉니다.
- 코사인 유사도는 새 문장과 기존 문서의 방향 유사성을 비교합니다.
- OOV, TF-IDF, bigram은 텍스트 분류 성능을 이해할 때 중요한 실험 축입니다.
